### **UTILIZATION OF LSTM ON MNIST DATASET**

In [ ]:
# imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:
# load dataset

train_path = pd.read_csv("dataset/mnist_train.csv")
test_path = pd.read_csv("dataset/mnist_test.csv")

print(f"Train Shape: {train_path.shape}")
print(f"Test Shape: {test_path.shape}")
print(f"First 5 records of the train dataset:\n{train_path.head()}")

print(f"Train CSV Columns: {train_path.columns.tolist()}")
print(f"Test CSV Columns: {test_path.columns.tolist()}")

In [ ]:
# preprocessing train set

X = train_path.iloc[:, 1:].values / 255.0
y = train_path.iloc[:, 0].values

# reshape for LSTM (sequence format)
X = X.reshape(-1, 28, 28)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

y_train = to_categorical(y_train, 10)
y_val   = to_categorical(y_val, 10)

print(f"Original X shape: {X.shape}")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")

# preprocessing test set

X_test = test_path.iloc[:, 1:].values / 255.0
X_test = X_test.reshape(-1, 28, 28)

print(f"X_test: {X_test.shape}")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# building the model

def build_lstm_model():
    model = Sequential([
        LSTM(128, input_shape=(28,28)),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_lstm_model()
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
# training the model

print("Training started for LSTM Model!")
model = build_lstm_model()
model_history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(12,10))

# accuracy, val_acc, loss and val_loss
plt.subplot(2,2,1)
plt.plot(model_history.history['accuracy'], label='Train Accuracy')
plt.plot(model_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model with No Augmentation - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# No Aug Loss
plt.subplot(2,2,2)
plt.plot(model_history.history['loss'], label='Train Loss')
plt.plot(model_history.history['val_loss'], label='Validation Loss')
plt.title('Model with No Augmentation - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# test prediction using the trained LSTM model
preds = model.predict(X_test)
y_pred = np.argmax(preds, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(X_test)+1),
    "Label": y_pred
})

submission.to_csv("lstm_submission.csv", index=False)
print(submission.head())